# Packages

In [1]:
import anndata

In [2]:
print(anndata.__version__)

0.12.0


In [2]:
from pathlib import Path
import scanpy as sc
import spatialdata as spd
from spatialdata.models import TableModel
from spatialdata import polygon_query

In [3]:
#Napari packages for tissue annotation
from qtpy.QtWidgets import QApplication
from napari_spatialdata import Interactive

In [4]:
import matplotlib.pyplot as plt
import spatialdata_plot  

# Functions

In [19]:
def make_sample_sdata(concat_sdata: spd.SpatialData, sample: str) -> spd.SpatialData:
    """
    Returns a SpatialData with:
      - the image for `sample`
      - the shapes for `sample` (cell boundaries)
      - the global table (segmentation_counts) unchanged
    """
    img_key = f"{sample}_hires_tissue_image"
    shp_key = f"{sample}_cell_boundaries"

    assert img_key in concat_sdata.images,  f"Missing image: {img_key}"
    assert shp_key in concat_sdata.shapes,  f"Missing shapes: {shp_key}"
    assert TABLE_KEY in concat_sdata.tables, "Missing table"

    sub = spd.SpatialData(
        images={img_key: concat_sdata.images[img_key]},
        shapes={shp_key: concat_sdata.shapes[shp_key]},
        tables={TABLE_KEY: concat_sdata.tables[TABLE_KEY]},
    )
    return sub


In [20]:
def crop_tissue(sub_sdata: spd.SpatialData, sample_id: str, tissue_name: str) -> spd.SpatialData:
    """
    sub_sdata: SpatialData for a single TMA (from make_sample_sdata)
    tissue_name: name of the ROI polygon in sub_sdata.shapes (e.g. "CyPSCA_1_1")

    Returns: SpatialData cropped to that polygon.
    """
    assert tissue_name in sub_sdata.shapes, f"ROI shape {tissue_name} not found in shapes."

    polygon = sub_sdata[tissue_name].geometry.iloc[0]
    cropped = polygon_query(
        sub_sdata,
        polygon=polygon,
        target_coordinate_system=CRS,
    )
    return cropped

In [21]:
def relabel_cropped_tissue(
    cropped_sdata: spd.SpatialData,
    sample_id: str,
    tissue_name: str,
) -> spd.SpatialData:
    """
    Take the cropped SpatialData (from crop_tissue) and:
      - rename image & shapes keys to tissue-specific names
      - add obs columns mouse, tissue
      - ensure region in the table matches the new shapes key
    """
    # Original keys in this cropped_sdata
    old_img_key = f"{sample_id}_hires_tissue_image"
    old_shp_key = f"{sample_id}_cell_boundaries"

    assert old_img_key in cropped_sdata.images, f"Expected image {old_img_key}"
    assert old_shp_key in cropped_sdata.shapes, f"Expected shapes {old_shp_key}"
    assert TABLE_KEY in cropped_sdata.tables, "Missing table in cropped_sdata"

    # New keys
    new_img_key = f"{tissue_name}_hires_tissue_image"
    new_shp_key = f"{tissue_name}_cell_boundaries"

    # Extract underlying data
    img_da      = cropped_sdata.images[old_img_key]
    shp_gdf     = cropped_sdata.shapes[old_shp_key]
    adata       = cropped_sdata.tables[TABLE_KEY].copy()

    # --- obs annotations ---
    
    adata.obs["TMA"]            = sample_id         # TMA ID (F07839, ...)
    adata.obs["mouse"]          = tissue_name       # the polygon / tumor / mouse label
    adata.obs["tissue"]         = tissue_name       # explicit tissue/ROI label
    adata.obs["condition"]      = tissue_name.split("_")[0] # Condition based on tissue name
    adata.obs["tumor_loc"]      = tissue_name.split("_")[1] # 1 = RT tumor; 2 = Abscopal tumor
    adata.obs["replicate_num"]  = tissue_name.split("_")[2] # Replicate number
    adata.obs["region"]         = new_shp_key
    adata.obs["region"]         = adata.obs["region"].astype("category")

    # Build a fresh SpatialData with tissue-level naming
    new_sdata = spd.SpatialData(
        images={new_img_key: img_da},
        shapes={
            new_shp_key: shp_gdf,
            # keep the ROI polygon as well, renamed explicitly as "_roi"
            f"{tissue_name}_roi": cropped_sdata.shapes[tissue_name],
        },
        tables={
            TABLE_KEY: TableModel.parse(
                adata,
                region=new_shp_key,
                region_key="region",
                instance_key="cell_id",
                overwrite_metadata=True,
            )
        },
    )

    return new_sdata


In [22]:
import matplotlib.pyplot as plt

def save_tissue_qc_plots(
    tissue_sd: spd.SpatialData,
    tissue_name: str,
    out_dir: Path,
) -> None:
    """
    For a tissue-level SpatialData (one image, one cell_boundaries, one ROI),
    save two sanity-check plots:
      1) H&E only
      2) H&E + cell boundaries overlay

    Files are saved in out_dir as:
      {tissue_name}_HE.png
      {tissue_name}_HE_cells.png
    """
    out_dir.mkdir(parents=True, exist_ok=True)

    # image key: first (and only) image
    img_key = list(tissue_sd.images.keys())[0]

    # shapes key: the one with '_cell_boundaries'
    shp_key_candidates = [k for k in tissue_sd.shapes.keys() if k.endswith("_cell_boundaries")]
    assert len(shp_key_candidates) == 1, f"Expected exactly one '*_cell_boundaries' in shapes, found {shp_key_candidates}"
    shp_key = shp_key_candidates[0]

    # --- Plot 1: H&E only ---
    fig, ax = plt.subplots(figsize=(6, 6))
    tissue_sd.pl.render_images(img_key).pl.show(
        coordinate_systems="downscale_to_hires",
        ax=ax,
    )
    ax.set_title(f"{tissue_name} - polygon_query H&E")
    he_path = out_dir / f"{tissue_name}_HE.png"
    fig.savefig(he_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

    # --- Plot 2: H&E + cell boundaries ---
    fig, ax = plt.subplots(figsize=(6, 6))
    (
        tissue_sd.pl.render_images(img_key)
        .pl.render_shapes(
            shp_key,
            color="lightgrey",
            fill_alpha=0.4,
            outline_alpha=0.4,
            outline_color="black",
            outline_width=0.2,
            method="matplotlib",
        )
        .pl.show(
            coordinate_systems="downscale_to_hires",
            ax=ax,
        )
    )
    ax.set_title(f"{tissue_name} - H&E + cell boundaries")
    he_cells_path = out_dir / f"{tissue_name}_HE_cells.png"
    fig.savefig(he_cells_path, dpi=300, bbox_inches="tight")
    plt.close(fig)


# Data Prep

In [23]:
# Main foldereed
zarr_folder = Path("/Users/janzules/Roselab/Spatial/CAR_T/data/zarrFiles")
concatenated  = zarr_folder / "concatenated_sdata"

sdata = spd.read_zarr(concatenated)

# These shouldn't ever change, but can be edited if we get more complex datasets in the future
TABLE_KEY = "segmentation_counts"
CRS       = "downscale_to_hires"

version mismatch: detected: RasterFormatV02, requested: FormatV04
/Users/janzules/miniforge3/envs/vishd_10x/lib/python3.11/site-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
version mismatch: detected: RasterFormatV02, requested: FormatV04
version mismatch: detected: RasterFormatV02, requested: FormatV04
version mismatch: detected: RasterFormatV02, requested: FormatV04


## Pre define tissue

In [24]:
TMA_tissues = {
    "F07839" : ["CyPSCA_1_1", "RTCyPSCA_1_4", "NoTx_2_2", "RTCyT72_2_1"],
    "F07840" : ["CyPSCA_1_2", "RTCyPSCA_2_4", "CyT72_1_4", "RTCyT72_2_4"],
    "F08542" : ["NoTx_1_4", "CyT72_2_3", "CyPSCA_2_4", "RTCyPSCA_1_3"],
    "F08543" : ["NoTx_2_4", "CyT72_1_2", "RTCyT72_1_1", "RTCyPSCA_2_3"],
}

# Manual Step

## Sub-sampling

In [25]:
# Tissue already processed: 

# Tissue to process
sample_id = "F08543"
sub_sdata = make_sample_sdata(sdata, sample_id)

/Users/janzules/miniforge3/envs/vishd_10x/lib/python3.11/site-packages/spatialdata/_core/spatialdata.py:184: UserWarning: The table is annotating 'F07839_cell_boundaries', which is not present in the SpatialData object.
  self.validate_table_in_spatialdata(v)
/Users/janzules/miniforge3/envs/vishd_10x/lib/python3.11/site-packages/spatialdata/_core/spatialdata.py:184: UserWarning: The table is annotating 'F07840_cell_boundaries', which is not present in the SpatialData object.
  self.validate_table_in_spatialdata(v)
/Users/janzules/miniforge3/envs/vishd_10x/lib/python3.11/site-packages/spatialdata/_core/spatialdata.py:184: UserWarning: The table is annotating 'F08542_cell_boundaries', which is not present in the SpatialData object.
  self.validate_table_in_spatialdata(v)


## Napari

In [26]:
# Each shape needs to be saved one at a time
app = QApplication.instance() or QApplication([])
viewer = Interactive(sub_sdata)
 
app.exec()  # blocks until you close the napari window

2025-12-04 17:05:44.003 | WARNING  | napari_spatialdata._viewer:__init__:57 - Due to Shift-L being used as shortcut in napari, it is being deprecated and might not link a new layer to an existing SpatialData object in the viewer. Please use ⌘-L on MacOS or else Ctrl-L.
/Users/janzules/miniforge3/envs/vishd_10x/lib/python3.11/functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
2025-12-04 17:05:53.722 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2025-12-04 17:05:53.723 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2025-12-04 18:06:37.703 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2025-12-04 18:06:37.706 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.


0

In [67]:
sub_sdata

SpatialData object
├── Images
│     └── 'F08543_hires_tissue_image': DataArray[cyx] (3, 5652, 6000)
├── Shapes
│     ├── 'CyT72_1_2': GeoDataFrame shape: (1, 1) (2D shapes)
│     ├── 'F08543_cell_boundaries': GeoDataFrame shape: (58897, 2) (2D shapes)
│     ├── 'NoTx_2_4': GeoDataFrame shape: (1, 1) (2D shapes)
│     ├── 'RTCyPSCA_2_3': GeoDataFrame shape: (1, 1) (2D shapes)
│     └── 'RTCyT72_1_1': GeoDataFrame shape: (1, 1) (2D shapes)
└── Tables
      └── 'segmentation_counts': AnnData (428822, 19059)
with coordinate systems:
    ▸ 'downscale_to_hires', with elements:
        F08543_hires_tissue_image (Images), CyT72_1_2 (Shapes), F08543_cell_boundaries (Shapes), NoTx_2_4 (Shapes), RTCyPSCA_2_3 (Shapes), RTCyT72_1_1 (Shapes)

## Per TMA processing

In [69]:
tissue_sdatas_for_this_sample = []

for tissue_name in TMA_tissues[sample_id]:
    print(f"Processing {sample_id} / {tissue_name}")

    cropped = crop_tissue(sub_sdata, sample_id, tissue_name)
    tissue_sd = relabel_cropped_tissue(cropped, sample_id, tissue_name)

    # Sanity check: number of cells and new obs columns
    adata_tissue = tissue_sd.tables[TABLE_KEY]
    print(
        f"  -> cells: {adata_tissue.n_obs}, "
        f"mouse={adata_tissue.obs['mouse'].unique()}, "
        f"tissue={adata_tissue.obs['tissue'].unique()}"
    )

    tissue_sdatas_for_this_sample.append(tissue_sd)

Processing F08543 / NoTx_2_4
  -> cells: 31434, mouse=['NoTx_2_4'], tissue=['NoTx_2_4']
Processing F08543 / CyT72_1_2
  -> cells: 5242, mouse=['CyT72_1_2'], tissue=['CyT72_1_2']
Processing F08543 / RTCyT72_1_1
  -> cells: 2747, mouse=['RTCyT72_1_1'], tissue=['RTCyT72_1_1']
Processing F08543 / RTCyPSCA_2_3
  -> cells: 19117, mouse=['RTCyPSCA_2_3'], tissue=['RTCyPSCA_2_3']


# Saving

## Individual Tissues

In [70]:
# import shutil
tissue_out = Path("/Users/janzules/Roselab/Spatial/CAR_T/data/zarrFiles/ByTissue")
tissue_out.mkdir(parents=True, exist_ok=True)

# directory for sanity-check images
tissue_img_out = tissue_out / "Tissue_images"
tissue_img_out.mkdir(parents=True, exist_ok=True)

for tissue_sd in tissue_sdatas_for_this_sample:
    # name by first (and only) tissue image key
    img_key = list(tissue_sd.images.keys())[0]
    tissue_name = img_key.replace("_hires_tissue_image", "")
    out_path = tissue_out / f"{sample_id}_{tissue_name}.zarr"
    print(f"Writing {out_path}")
    tissue_sd.write(out_path, overwrite=True)

    # save sanity-check plots for this tissue
    save_tissue_qc_plots(
        tissue_sd=tissue_sd,
        tissue_name=tissue_name,
        out_dir=tissue_img_out,
    )


Writing /Users/janzules/Roselab/Spatial/CAR_T/data/zarrFiles/ByTissue/F08543_NoTx_2_4.zarr
INFO     The SpatialData object is not self-contained (i.e. it contains some elements that are Dask-backed from    
         locations outside /Users/janzules/Roselab/Spatial/CAR_T/data/zarrFiles/ByTissue/F08543_NoTx_2_4.zarr).    
         Please see the documentation of `is_self_contained()` to understand the implications of working with      
         SpatialData objects that are not self-contained.                                                          
INFO     The Zarr backing store has been changed from None the new file path:                                      
         /Users/janzules/Roselab/Spatial/CAR_T/data/zarrFiles/ByTissue/F08543_NoTx_2_4.zarr                        
INFO     Rasterizing image for faster rendering.                                                                   
INFO     Value for parameter 'color' appears to be a color, using it as such.                    

## Complete Datasets concatenated

In [ ]:
zarr_folder

In [71]:
# Collect all tissue-level zarr paths
# tissue_out = Path("/Volumes/workl/Jona/spatial/second_run_output/tissues")

tissue_zarr_paths = sorted(tissue_out.glob("*.zarr"))
print("Found tissue-level zarrs:", len(tissue_zarr_paths))

tissue_sdatas = []
for p in tissue_zarr_paths:
    tissue_sdatas.append(spd.read_zarr(p))

# Concatenate
tissue_concat = spd.concatenate(tissue_sdatas, concatenate_tables=True)

print(tissue_concat)
tissue_concat_out = zarr_folder / "concatenated_tissues_sdata_second_run"
tissue_concat.write(tissue_concat_out, overwrite=True)


version mismatch: detected: RasterFormatV02, requested: FormatV04


Found tissue-level zarrs: 16


/Users/janzules/miniforge3/envs/vishd_10x/lib/python3.11/site-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
version mismatch: detected: RasterFormatV02, requested: FormatV04
/Users/janzules/miniforge3/envs/vishd_10x/lib/python3.11/site-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
version mismatch: detected: RasterFormatV02, requested: FormatV04
/Users/janzules/miniforge3/envs/vishd_10x/lib/python3.11/site-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
version mismatch: detected: RasterFormatV02, requested: FormatV04
/Users/janzules/miniforge3/envs/vishd_10x/lib/python3.11/site-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  co

SpatialData object
├── Images
│     ├── 'CyPSCA_1_1_hires_tissue_image': DataArray[cyx] (3, 2645, 2706)
│     ├── 'CyPSCA_1_2_hires_tissue_image': DataArray[cyx] (3, 2348, 2448)
│     ├── 'CyPSCA_2_4_hires_tissue_image': DataArray[cyx] (3, 2445, 2505)
│     ├── 'CyT72_1_2_hires_tissue_image': DataArray[cyx] (3, 2202, 2308)
│     ├── 'CyT72_1_4_hires_tissue_image': DataArray[cyx] (3, 2224, 2060)
│     ├── 'CyT72_2_3_hires_tissue_image': DataArray[cyx] (3, 2480, 2363)
│     ├── 'NoTx_1_4_hires_tissue_image': DataArray[cyx] (3, 2600, 1799)
│     ├── 'NoTx_2_2_hires_tissue_image': DataArray[cyx] (3, 2668, 2533)
│     ├── 'NoTx_2_4_hires_tissue_image': DataArray[cyx] (3, 2494, 2542)
│     ├── 'RTCyPSCA_1_3_hires_tissue_image': DataArray[cyx] (3, 2355, 2630)
│     ├── 'RTCyPSCA_1_4_hires_tissue_image': DataArray[cyx] (3, 2658, 2085)
│     ├── 'RTCyPSCA_2_3_hires_tissue_image': DataArray[cyx] (3, 2897, 2837)
│     ├── 'RTCyPSCA_2_4_hires_tissue_image': DataArray[cyx] (3, 2513, 2669)
│     ├──

In [ ]:
tissue_concat['segmentation_counts']

# Further Processing data

In [12]:
tissue_concat_zarr = Path("/Users/janzules/Roselab/Spatial/CAR_T/data/zarrFiles/concatenated_tissues_sdata_second_run")

sdata = spd.read_zarr(tissue_concat_zarr)

version mismatch: detected: RasterFormatV02, requested: FormatV04
/Users/janzules/miniforge3/envs/vishd_10x/lib/python3.11/site-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
version mismatch: detected: RasterFormatV02, requested: FormatV04
version mismatch: detected: RasterFormatV02, requested: FormatV04
version mismatch: detected: RasterFormatV02, requested: FormatV04
version mismatch: detected: RasterFormatV02, requested: FormatV04
version mismatch: detected: RasterFormatV02, requested: FormatV04
version mismatch: detected: RasterFormatV02, requested: FormatV04
version mismatch: detected: RasterFormatV02, requested: FormatV04
version mismatch: detected: RasterFormatV02, requested: FormatV04
version mismatch: detected: RasterFormatV02, requested: FormatV04
version mismatch: detected: RasterFormatV02, requested: FormatV04
version mismatch: detected: RasterFormatV02, requested: 

In [13]:
TABLE_KEY = "segmentation_counts"

adata = sdata.tables[TABLE_KEY]
print("n_obs before:", adata.n_obs)

# Find all rows with duplicated cell_id (both entries)
dup_mask = adata.obs["cell_id"].duplicated(keep=False)
dups = adata.obs[dup_mask].copy()

print("Number of duplicated rows:", dups.shape[0])
print("Number of unique duplicated cell_ids:", dups["cell_id"].nunique())

# These are the cell_ids we will drop entirely
dup_cell_ids = dups["cell_id"].unique()
print("cell_ids to drop:", dup_cell_ids)


n_obs before: 427318
Number of duplicated rows: 22
Number of unique duplicated cell_ids: 11
cell_ids to drop: ['F07840_cellid_000095096-1' 'F07840_cellid_000097531-1'
 'F07840_cellid_000099534-1' 'F07840_cellid_000104740-1'
 'F07840_cellid_000106594-1' 'F07840_cellid_000107421-1'
 'F08543_cellid_000000240-1' 'F08543_cellid_000000710-1'
 'F08543_cellid_000001212-1' 'F08543_cellid_000001646-1'
 'F08543_cellid_000005964-1']


In [15]:
import pandas as pd
import numpy as np

# --- 2.1 Drop from shapes ---

for shp_name, gdf in list(sdata.shapes.items()):
    if not shp_name.endswith("_cell_boundaries"):
        continue  # skip ROI layers

    before = gdf.shape[0]
    gdf_filtered = gdf[~gdf.index.isin(dup_cell_ids)].copy()
    after = gdf_filtered.shape[0]
    print(f"{shp_name}: {before} -> {after} (dropped {before - after})")

    sdata.shapes[shp_name] = gdf_filtered

# --- 2.2 Drop from AnnData table ---

adata = sdata.tables[TABLE_KEY]
before = adata.n_obs
keep_mask = ~adata.obs["cell_id"].isin(dup_cell_ids)
adata = adata[keep_mask].copy()
after = adata.n_obs
print("AnnData rows:", before, "->", after, "(dropped", before - after, ")")

# sanity: cell_id duplicates should now be gone
assert not adata.obs["cell_id"].duplicated().any(), "Still have duplicated cell_id in adata.obs"

# --- 2.3 Recompute centroids from cleaned shapes ---

centroid_frames = []

for shp_name, gdf in sdata.shapes.items():
    if not shp_name.endswith("_cell_boundaries"):
        continue

    geom = gdf.geometry
    centroids = geom.centroid

    df = pd.DataFrame(
        {
            "cell_id": gdf.index.astype(str),
            "x": centroids.x.values,
            "y": centroids.y.values,
        }
    ).set_index("cell_id")

    print(f"{shp_name}: {len(df)} centroids")
    centroid_frames.append(df)

centroids_all = pd.concat(centroid_frames, axis=0)
print("Total centroids:", len(centroids_all))

# centroids_all index should now be unique
assert not centroids_all.index.duplicated().any(), "centroids_all index still has duplicates"

# --- 2.4 Align centroids with adata.obs["cell_id"] and set obsm["spatial"] ---

cell_ids = adata.obs["cell_id"].astype(str)

missing = set(cell_ids) - set(centroids_all.index)
if missing:
    print(f"WARNING: {len(missing)} cell_ids in adata but missing in shapes/centroids.")
    print("Example missing IDs:", list(sorted(missing))[:10])

centroids_ordered = centroids_all.reindex(cell_ids)

if centroids_ordered.isna().any().any():
    raise ValueError("Some centroids are NaN – check missing cell_ids before proceeding.")

coords = centroids_ordered[["x", "y"]].to_numpy(dtype=float)
print("coords shape:", coords.shape)  # should be (adata.n_obs, 2)

adata.obsm["spatial"] = coords

# Reattach updated adata to SpatialData
sdata.tables[TABLE_KEY] = adata  # in-memory overwrite


CyPSCA_1_1_cell_boundaries: 45836 -> 45836 (dropped 0)
CyPSCA_1_2_cell_boundaries: 22650 -> 22650 (dropped 0)
CyPSCA_2_4_cell_boundaries: 16127 -> 16127 (dropped 0)
CyT72_1_2_cell_boundaries: 5242 -> 5237 (dropped 5)
CyT72_1_4_cell_boundaries: 26153 -> 26153 (dropped 0)
CyT72_2_3_cell_boundaries: 17859 -> 17859 (dropped 0)
NoTx_1_4_cell_boundaries: 14820 -> 14820 (dropped 0)
NoTx_2_2_cell_boundaries: 65864 -> 65864 (dropped 0)
NoTx_2_4_cell_boundaries: 31434 -> 31429 (dropped 5)
RTCyPSCA_1_3_cell_boundaries: 17083 -> 17083 (dropped 0)
RTCyPSCA_1_4_cell_boundaries: 28353 -> 28353 (dropped 0)
RTCyPSCA_2_3_cell_boundaries: 19117 -> 19117 (dropped 0)
RTCyPSCA_2_4_cell_boundaries: 27950 -> 27944 (dropped 6)
RTCyT72_1_1_cell_boundaries: 2747 -> 2747 (dropped 0)
RTCyT72_2_1_cell_boundaries: 38821 -> 38821 (dropped 0)
RTCyT72_2_4_cell_boundaries: 47262 -> 47256 (dropped 6)


/Users/janzules/miniforge3/envs/vishd_10x/lib/python3.11/site-packages/spatialdata/_core/_elements.py:105: UserWarning: Key `CyPSCA_1_1_cell_boundaries` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)
/Users/janzules/miniforge3/envs/vishd_10x/lib/python3.11/site-packages/spatialdata/_core/_elements.py:105: UserWarning: Key `CyPSCA_1_2_cell_boundaries` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)
/Users/janzules/miniforge3/envs/vishd_10x/lib/python3.11/site-packages/spatialdata/_core/_elements.py:105: UserWarning: Key `CyPSCA_2_4_cell_boundaries` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)
/Users/janzules/miniforge3/envs/vishd_10x/lib/python3.11/site-packages/spatialdata/_core/_elements.py:105: UserWarning: Key `CyT72_1_2_cell_boundaries` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)
/

AnnData rows: 427318 -> 427296 (dropped 22 )
CyPSCA_1_1_cell_boundaries: 45836 centroids
CyPSCA_1_2_cell_boundaries: 22650 centroids
CyPSCA_2_4_cell_boundaries: 16127 centroids
CyT72_1_2_cell_boundaries: 5237 centroids
CyT72_1_4_cell_boundaries: 26153 centroids
CyT72_2_3_cell_boundaries: 17859 centroids
NoTx_1_4_cell_boundaries: 14820 centroids
NoTx_2_2_cell_boundaries: 65864 centroids
NoTx_2_4_cell_boundaries: 31429 centroids
RTCyPSCA_1_3_cell_boundaries: 17083 centroids
RTCyPSCA_1_4_cell_boundaries: 28353 centroids
RTCyPSCA_2_3_cell_boundaries: 19117 centroids
RTCyPSCA_2_4_cell_boundaries: 27944 centroids
RTCyT72_1_1_cell_boundaries: 2747 centroids
RTCyT72_2_1_cell_boundaries: 38821 centroids
RTCyT72_2_4_cell_boundaries: 47256 centroids
Total centroids: 427296
coords shape: (427296, 2)


/Users/janzules/miniforge3/envs/vishd_10x/lib/python3.11/site-packages/spatialdata/_core/_elements.py:125: UserWarning: Key `segmentation_counts` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)


In [18]:
# out_path = zarr_folder / "concatenated_tissues_sdata_second_run_clean"
tissue_concat_zarr_clean = Path("/Users/janzules/Roselab/Spatial/CAR_T/data/zarrFiles/concatenated_tissues_sdata_second_run_clean")
print("Writing", tissue_concat_zarr_clean)
sdata.write(tissue_concat_zarr_clean, overwrite=True)

Writing /Users/janzules/Roselab/Spatial/CAR_T/data/zarrFiles/concatenated_tissues_sdata_second_run_clean
INFO     The SpatialData object is not self-contained (i.e. it contains some elements that are Dask-backed from    
         locations outside                                                                                         
         /Users/janzules/Roselab/Spatial/CAR_T/data/zarrFiles/concatenated_tissues_sdata_second_run_clean). Please 
         see the documentation of `is_self_contained()` to understand the implications of working with SpatialData 
         objects that are not self-contained.                                                                      
INFO     The Zarr backing store has been changed from                                                              
         /Users/janzules/Roselab/Spatial/CAR_T/data/zarrFiles/concatenated_tissues_sdata_second_run the new file   
         path: /Users/janzules/Roselab/Spatial/CAR_T/data/zarrFiles/concatenated_ti

# Testing

In [6]:
import numpy as np
import pandas as pd

TABLE_KEY = "segmentation_counts"

adata = sdata.tables[TABLE_KEY]
print(adata)

print("obs columns:", adata.obs.columns.tolist())
print("n_obs:", adata.n_obs)
print("unique regions in table:", adata.obs["region"].unique())


AnnData object with n_obs × n_vars = 427318 × 19059
    obs: 'sample', 'cell_id', 'region', 'TMA', 'mouse', 'tissue', 'condition', 'tumor_loc', 'replicate_num'
    uns: 'spatialdata_attrs'
obs columns: ['sample', 'cell_id', 'region', 'TMA', 'mouse', 'tissue', 'condition', 'tumor_loc', 'replicate_num']
n_obs: 427318
unique regions in table: ['CyPSCA_1_1_cell_boundaries', 'CyPSCA_1_2_cell_boundaries', 'CyPSCA_2_4_cell_boundaries', 'CyT72_1_4_cell_boundaries', 'CyT72_2_3_cell_boundaries', ..., 'RTCyT72_2_4_cell_boundaries', 'CyT72_1_2_cell_boundaries', 'NoTx_2_4_cell_boundaries', 'RTCyPSCA_2_3_cell_boundaries', 'RTCyT72_1_1_cell_boundaries']
Length: 16
Categories (16, object): ['CyPSCA_1_1_cell_boundaries', 'CyPSCA_1_2_cell_boundaries', 'CyPSCA_2_4_cell_boundaries', 'CyT72_1_2_cell_boundaries', ..., 'RTCyPSCA_2_4_cell_boundaries', 'RTCyT72_1_1_cell_boundaries', 'RTCyT72_2_1_cell_boundaries', 'RTCyT72_2_4_cell_boundaries']


In [7]:
from shapely.geometry import Polygon  # just to be explicit; geometry is already shapely

centroid_frames = []

for shp_name, gdf in sdata.shapes.items():
    if not shp_name.endswith("_cell_boundaries"):
        continue  # skip ROI layers

    # gdf.index should be cell_ids; geometry column holds polygons
    geom = gdf.geometry
    centroids = geom.centroid  # GeoSeries of Point objects

    df = pd.DataFrame(
        {
            "cell_id": gdf.index.astype(str),
            "x": centroids.x.values,
            "y": centroids.y.values,
        }
    ).set_index("cell_id")

    print(f"{shp_name}: {len(df)} centroids")
    centroid_frames.append(df)

centroids_all = pd.concat(centroid_frames, axis=0)
print("Total centroids:", len(centroids_all))
print("First few rows:")
print(centroids_all.head())


CyPSCA_1_1_cell_boundaries: 45836 centroids
CyPSCA_1_2_cell_boundaries: 22650 centroids
CyPSCA_2_4_cell_boundaries: 16127 centroids
CyT72_1_2_cell_boundaries: 5242 centroids
CyT72_1_4_cell_boundaries: 26153 centroids
CyT72_2_3_cell_boundaries: 17859 centroids
NoTx_1_4_cell_boundaries: 14820 centroids
NoTx_2_2_cell_boundaries: 65864 centroids
NoTx_2_4_cell_boundaries: 31434 centroids
RTCyPSCA_1_3_cell_boundaries: 17083 centroids
RTCyPSCA_1_4_cell_boundaries: 28353 centroids
RTCyPSCA_2_3_cell_boundaries: 19117 centroids
RTCyPSCA_2_4_cell_boundaries: 27950 centroids
RTCyT72_1_1_cell_boundaries: 2747 centroids
RTCyT72_2_1_cell_boundaries: 38821 centroids
RTCyT72_2_4_cell_boundaries: 47262 centroids
Total centroids: 427318
First few rows:
                                     x            y
cell_id                                            
F07839_cellid_000000001-1  2243.106273  3877.965410
F07839_cellid_000000002-1  3007.457085  1910.707490
F07839_cellid_000000004-1  1196.668877  2384.424

In [8]:
# Sanity check: ensure we have centroids for all cells in adata
cell_ids = adata.obs["cell_id"].astype(str)
missing = set(cell_ids) - set(centroids_all.index)

if missing:
    print(f"WARNING: {len(missing)} cell_ids present in adata but missing in shapes/centroids.")
    # If this is non-zero, we can inspect a few:
    print("Example missing IDs:", list(sorted(missing))[:10])

# Reindex centroids in the exact order of adata.obs["cell_id"]
centroids_ordered = centroids_all.reindex(cell_ids)

# If you want to be strict, you can assert no NaNs:
if centroids_ordered.isna().any().any():
    raise ValueError("Some centroids are NaN – check missing cell_ids before proceeding.")

coords = centroids_ordered[["x", "y"]].to_numpy(dtype=float)
print("coords shape:", coords.shape)  # should be (adata.n_obs, 2)

# Attach to AnnData
adata.obsm["spatial"] = coords
sdata.tables[TABLE_KEY] = adata  # reassign back to SpatialData

# Quick sanity check on a few rows
print("Example spatial coords:")
print(adata.obsm["spatial"][:5])


ValueError: cannot reindex on an axis with duplicate labels

In [9]:
# Sanity check: ensure we have centroids for all cells in adata
cell_ids = adata.obs["cell_id"].astype(str)
missing = set(cell_ids) - set(centroids_all.index)

if missing:
    print(f"WARNING: {len(missing)} cell_ids present in adata but missing in shapes/centroids.")
    print("Example missing IDs:", list(sorted(missing))[:10])

# --- New: handle duplicate cell_ids in centroids_all ---
if centroids_all.index.duplicated().any():
    n_dup = centroids_all.index.duplicated().sum()
    print(f"Found {n_dup} duplicated cell_id entries in centroids_all; collapsing by mean (x, y).")
    # Group by cell_id (index) and take the mean centroid per cell_id
    centroids_all = (
        centroids_all
        .groupby(level=0)[["x", "y"]]
        .mean()
    )

# Optional: check that cell_ids in adata are unique (they should be)
if adata.obs["cell_id"].duplicated().any():
    n_dup_cells = adata.obs["cell_id"].duplicated().sum()
    print(f"WARNING: {n_dup_cells} duplicated 'cell_id' entries in adata.obs['cell_id'].")

# Reindex centroids in the exact order of adata.obs["cell_id"]
centroids_ordered = centroids_all.reindex(cell_ids)

# If you want to be strict, you can assert no NaNs:
if centroids_ordered.isna().any().any():
    raise ValueError("Some centroids are NaN – check missing cell_ids before proceeding.")

coords = centroids_ordered[["x", "y"]].to_numpy(dtype=float)
print("coords shape:", coords.shape)  # should be (adata.n_obs, 2)

# Attach to AnnData
adata.obsm["spatial"] = coords
sdata.tables[TABLE_KEY] = adata  # reassign back to SpatialData

# Quick sanity check on a few rows
print("Example spatial coords:")
print(adata.obsm["spatial"][:5])


Found 11 duplicated cell_id entries in centroids_all; collapsing by mean (x, y).
coords shape: (427318, 2)
Example spatial coords:
[[2243.10627289 3877.96541023]
 [3007.45708502 1910.70748988]
 [1196.6688773  2384.42431694]
 [2793.87684933 2771.57125605]
 [3933.94486225 2691.92505585]]


/Users/janzules/miniforge3/envs/vishd_10x/lib/python3.11/site-packages/spatialdata/_core/_elements.py:125: UserWarning: Key `segmentation_counts` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)


In [11]:
# all duplicated cell_ids (both instances)
dup_mask = adata.obs["cell_id"].duplicated(keep=False)
dups = adata.obs[dup_mask].copy()

print("Number of duplicated rows:", dups.shape[0])
print("Number of unique duplicated cell_ids:", dups["cell_id"].nunique())

# See which tissues / regions they belong to
cols_to_view = ["cell_id", "TMA", "tissue", "region"]
print(dups[cols_to_view].sort_values("cell_id"))


Number of duplicated rows: 22
Number of unique duplicated cell_ids: 11
                                               cell_id     TMA        tissue  \
F07840_cellid_000095096-1    F07840_cellid_000095096-1  F07840  RTCyPSCA_2_4   
F07840_cellid_000095096-1-1  F07840_cellid_000095096-1  F07840   RTCyT72_2_4   
F07840_cellid_000097531-1    F07840_cellid_000097531-1  F07840  RTCyPSCA_2_4   
F07840_cellid_000097531-1-1  F07840_cellid_000097531-1  F07840   RTCyT72_2_4   
F07840_cellid_000099534-1    F07840_cellid_000099534-1  F07840  RTCyPSCA_2_4   
F07840_cellid_000099534-1-1  F07840_cellid_000099534-1  F07840   RTCyT72_2_4   
F07840_cellid_000104740-1    F07840_cellid_000104740-1  F07840  RTCyPSCA_2_4   
F07840_cellid_000104740-1-1  F07840_cellid_000104740-1  F07840   RTCyT72_2_4   
F07840_cellid_000106594-1-1  F07840_cellid_000106594-1  F07840   RTCyT72_2_4   
F07840_cellid_000106594-1    F07840_cellid_000106594-1  F07840  RTCyPSCA_2_4   
F07840_cellid_000107421-1-1  F07840_cellid_000107

In [10]:
adata.obsm["spatial"]
# array([[x0, y0],
#        [x1, y1],
#        ...,
#        [xn, yn]])


array([[2243.10627289, 3877.96541023],
       [3007.45708502, 1910.70748988],
       [1196.6688773 , 2384.42431694],
       ...,
       [6390.63778089, 6722.43893764],
       [6425.88500703, 6550.06110843],
       [6314.0503469 , 6782.51023321]])